## RLHF-style safety tuning.

This lab is teaching you how to take an already instruction-tuned LLM and further train it to produce less harmful or toxic responses. The key idea is that a separate toxicity/hate-speech classifier acts like a judge, and PPO updates the model so its answers score lower on toxicity. Use this idea when you want to improve model behavior with feedback signals, but not when simple prompt changes or filtering are enough. For the interview assistant, this same pattern could help you score answers for quality, clarity, correctness, or professionalism, then improve the assistant based on those scores.



In [1]:
#to verify that there is enough compute sources in this project:

import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

GPU available: True
GPU name: Tesla T4


In [2]:
# Cell 2 — REPLACE with this version
import sys, importlib

# Force uninstall old versions
%pip uninstall -y peft trl transformers datasets evaluate accelerate safetensors sentence-transformers

# Install pinned versions that are compatible with each other
%pip install -q \
    transformers==4.38.2 \
    datasets==2.17.0 \
    evaluate==0.4.0 \
    rouge_score==0.1.2 \
    accelerate==0.28.0 \
    safetensors \
    peft==0.9.0 \
    sentence-transformers==2.7.0

# Install the pinned TRL version that works with transformers 4.38.2
%pip install -q --no-deps git+https://github.com/lvwerra/trl.git@25fa1bd

print("All packages installed")

Found existing installation: peft 0.9.0
Uninstalling peft-0.9.0:
  Successfully uninstalled peft-0.9.0
Found existing installation: trl 0.4.2.dev0
Uninstalling trl-0.4.2.dev0:
  Successfully uninstalled trl-0.4.2.dev0
Found existing installation: transformers 4.38.2
Uninstalling transformers-4.38.2:
  Successfully uninstalled transformers-4.38.2
Found existing installation: datasets 2.17.0
Uninstalling datasets-2.17.0:
  Successfully uninstalled datasets-2.17.0
Found existing installation: evaluate 0.4.0
Uninstalling evaluate-0.4.0:
  Successfully uninstalled evaluate-0.4.0
Found existing installation: accelerate 0.28.0
Uninstalling accelerate-0.28.0:
  Successfully uninstalled accelerate-0.28.0
Found existing installation: safetensors 0.8.0
Uninstalling safetensors-0.8.0:
  Successfully uninstalled safetensors-0.8.0
Found existing installation: sentence-transformers 2.7.0
Uninstalling sentence-transformers-2.7.0:
  Successfully uninstalled sentence-transformers-2.7.0
  Preparing metad

In [3]:
# Run this as a standalone cell AFTER Cell 2 finishes
# It forces Colab to restart the Python kernel so old packages are cleared from memory
#import os
#os.kill(os.getpid(), 9)

In [4]:
#%pip install numpy==1.26.4

In [5]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM, GenerationConfig
from datasets import load_dataset
from peft import PeftModel, PeftConfig, LoraConfig, TaskType

from trl import PPOTrainer, PPOConfig, AutoModelForSeq2SeqLMWithValueHead
from trl import create_reference_model
from trl.core import LengthSampler

import torch
import evaluate
import numpy as np
import pandas as pd

from tqdm import tqdm
tqdm.pandas()

# NEW: add sentence-transformers import
from sentence_transformers import SentenceTransformer, util

Load dataset and LLM
I am going to continue experimenting with the DataScienceInterviewQuestions Hugging Face dataset https://huggingface.co/datasets/mjphayes/machine_learning_questions/viewer & https://huggingface.co/datasets/UdayG01/DataScienceInterviewQuestions. It contains 636 questions with answers combining both datasets.


In [6]:
from datasets import load_dataset, concatenate_datasets, DatasetDict, Dataset
import pandas as pd

# Dataset 1: mjphayes — already lowercase question / answer
ds1 = load_dataset("mjphayes/machine_learning_questions")
if "__index_level_0__" in ds1["train"].column_names:
    ds1 = ds1.remove_columns("__index_level_0__")

# Dataset 2: UdayG01 — capitalized, so rename to lowercase
ds2 = load_dataset("UdayG01/DataScienceInterviewQuestions")
ds2 = ds2.rename_columns({"Question": "question", "Answer": "answer"})

keep = ["question", "answer"]

# Use BOTH splits of mjphayes (train 508 + test 128 = 636)
ds1_all = concatenate_datasets([
    ds1["train"].select_columns(keep),
    ds1["test"].select_columns(keep),
])

# UdayG01 train (47)
ds2_train = ds2["train"].select_columns(keep)

# Merge into one pool (636 + 47 = 683)  <-- this line was missing
combined = concatenate_datasets([ds1_all, ds2_train])
print(combined)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/508 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/128 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/47 [00:00<?, ? examples/s]

Dataset({
    features: ['question', 'answer'],
    num_rows: 683
})


In [7]:
# Remove duplicate questions BEFORE splitting (avoids leakage)
df = combined.to_pandas()
before = len(df)
df = df.drop_duplicates(subset=["question"]).reset_index(drop=True)
print(f"removed {before - len(df)} duplicate questions, {len(df)} remain")
combined = Dataset.from_pandas(df, preserve_index=False)

removed 32 duplicate questions, 651 remain


In [8]:
split_1 = combined.train_test_split(test_size=0.30, seed=42)        # 70% train
split_2 = split_1["test"].train_test_split(test_size=0.5, seed=42)  # 15% val, 15% test

dataset = DatasetDict({
    "train": split_1["train"],
    "validation": split_2["train"],
    "test": split_2["test"],
})
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 455
    })
    validation: Dataset({
        features: ['question', 'answer'],
        num_rows: 98
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 98
    })
})


Load FLAN-T5 Model, Prepare Reward Model and Toxicity Evaluator

In [9]:
model_name = "google/flan-t5-base"

dataset_original = dataset   # your DatasetDict with train/validation/test splits

dataset_original

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 455
    })
    validation: Dataset({
        features: ['question', 'answer'],
        num_rows: 98
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 98
    })
})

The next step will be to preprocess the dataset. You will take only a part of it, then filter the dialogues of a particular length (just to make those examples long enough and, at the same time, easy to read). Then wrap each dialogue with the instruction and tokenize the prompts. Save the token ids in the field input_ids and decoded version of the prompts in the field query.

You could do that all step by step in the cell below, but it is a good habit to organize that all in a function build_dataset:

In [10]:
from transformers import AutoTokenizer

def build_dataset(model_name, dataset, input_min_text_length, input_max_text_length):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def filter_length(x):
        return (
            len(x["question"]) > input_min_text_length
            and len(x["answer"]) <= input_max_text_length
        )

    def tokenize(sample):
        prompt = f"""
 Answer the following data science interview question clearly and concisely.\n\n

{sample["question"]}

Answer:
"""
        sample["input_ids"] = tokenizer.encode(prompt)
        sample["query"] = tokenizer.decode(sample["input_ids"])
        return sample

    dataset = dataset.filter(filter_length, batched=False)
    dataset = dataset.map(tokenize, batched=False)
    dataset.set_format(type="torch")

    return dataset

In [11]:
dataset = build_dataset(
    model_name=model_name,
    dataset=dataset,
    input_min_text_length=20,
    input_max_text_length=1000
)

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

Filter:   0%|          | 0/98 [00:00<?, ? examples/s]

Map:   0%|          | 0/455 [00:00<?, ? examples/s]

Map:   0%|          | 0/96 [00:00<?, ? examples/s]

Map:   0%|          | 0/96 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'input_ids', 'query'],
        num_rows: 455
    })
    validation: Dataset({
        features: ['question', 'answer', 'input_ids', 'query'],
        num_rows: 96
    })
    test: Dataset({
        features: ['question', 'answer', 'input_ids', 'query'],
        num_rows: 96
    })
})


In the Data assistant fintuning part of the project I fine-tuned the PEFT model with Question/Answers. The training in the notebook was done on a subset of data.  Downloaded the checkpoint of the fully trained PEFT model.

Let's load the same model checkpoint here:

In [12]:
from google.colab import drive
drive.mount("/content/drive")

!ls -la /content/drive/MyDrive/peft-ds-interview-checkpoint-local

Mounted at /content/drive
total 16222
-rw------- 1 root root      608 Jun 11 14:17 adapter_config_backup.json
-rw------- 1 root root      608 Jun 11 14:17 adapter_config.json
-rw------- 1 root root 14176016 Jun  9 21:11 adapter_model.safetensors
-rw------- 1 root root     5162 Jun 10 22:19 README.md
-rw------- 1 root root     2381 Jun 10 22:19 tokenizer_config.json
-rw------- 1 root root  2424069 Jun 10 22:19 tokenizer.json


In [13]:
import json
import inspect
from peft import LoraConfig

peft_model_path = "/content/drive/MyDrive/peft-ds-interview-checkpoint-local"
config_path = peft_model_path + "/adapter_config.json"

# Backup original config once
backup_path = peft_model_path + "/adapter_config_backup.json"

with open(config_path, "r") as f:
    config = json.load(f)

with open(backup_path, "w") as f:
    json.dump(config, f, indent=2)

# Keep only keys your installed LoraConfig understands
valid_keys = set(inspect.signature(LoraConfig.__init__).parameters.keys())
valid_keys.discard("self")

clean_config = {k: v for k, v in config.items() if k in valid_keys}

# Keep PEFT type if it exists
if "peft_type" in config:
    clean_config["peft_type"] = config["peft_type"]

with open(config_path, "w") as f:
    json.dump(clean_config, f, indent=2)

print("Cleaned adapter_config.json")
print("Removed keys:", sorted(set(config.keys()) - set(clean_config.keys())))

Cleaned adapter_config.json
Removed keys: []


In [14]:
# Run this in Lab 3 before doing anything else
import os

local_path = "./peft-ds-interview-checkpoint-local"
drive_path = "/content/drive/MyDrive/peft-ds-interview-checkpoint-local"

print("Local exists:", os.path.exists(local_path))
print("Drive exists:", os.path.exists(drive_path))

# If Drive exists, check when it was last modified
if os.path.exists(drive_path):
    import datetime
    mtime = os.path.getmtime(drive_path + "/adapter_config.json")
    print("Drive checkpoint modified:", datetime.datetime.fromtimestamp(mtime))

Local exists: False
Drive exists: True
Drive checkpoint modified: 2026-06-11 15:17:10


In [15]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel
import torch

peft_model_path = "/content/drive/MyDrive/peft-ds-interview-checkpoint-local"

peft_model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-base",
    torch_dtype=torch.float16
).to("cuda")

peft_model = PeftModel.from_pretrained(peft_model, peft_model_path).to("cuda")

tokenizer = AutoTokenizer.from_pretrained(
    "google/flan-t5-base",
    use_fast=False
)

peft_model.eval()

print("PEFT model loaded")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


PEFT model loaded


Prepare a function to pull out the number of model parameters (it is the same as in the previous lab):

Use this after loading a model, applying LoRA, or preparing a model for PPO so you can verify that only the intended parameters are trainable. Do not use it as a quality metric because it only tells you model size/training setup, not whether the model answers well or became less toxic.


In [16]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f"\ntrainable model parameters: {trainable_model_params}\nall model parameters: {all_model_params}\npercentage of trainable model parameters: {100 * trainable_model_params / all_model_params:.2f}%"

Add the adapter to the original FLAN-T5 model. In the previous lab you were adding the fully trained adapter only for inferences, so there was no need to pass LoRA configurations doing that. Now you need to pass them to the constructed PEFT model, also putting is_trainable=True.


In [17]:
lora_config = LoraConfig(
    r=32, # Rank
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM # FLAN-T5
)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name,
                                              torch_dtype=torch.bfloat16).to("cuda")

peft_model = PeftModel.from_pretrained(model,
                                       "/content/drive/MyDrive/peft-ds-interview-checkpoint-local",
                                       lora_config=lora_config,
                                       torch_dtype=torch.bfloat16,
                                       device_map="auto",
                                       is_trainable=True).to("cuda")

print(f'PEFT model parameters to be updated:\n{print_number_of_trainable_model_parameters(peft_model)}\n')

PEFT model parameters to be updated:

trainable model parameters: 3538944
all model parameters: 251116800
percentage of trainable model parameters: 1.41%



**The PEFT model is no longer frozen**. Only **1.41%** of the full model’s parameters are trainable, which is the point of PEFT/LoRA: you update a small adapter instead of retraining the whole model.

PPO needs trainable parameters because it will adjust the model based on reward scores from the toxicity classifier. Since the trainable parameters are now above 0, the adapter can actually be updated during PPO training.

For the interview assistant project:
- This same setup would let you start with your instruction-tuned data science interview model, then continue training only the small adapter to improve answer quality, reduce bad responses, or make the assistant better at explaining ML/statistics concepts.

For this project I am  preparing to fine-tune the LLM using **Reinforcement Learning (RL)**.At this stage, we just need to prepare the Proximal Policy Optimization (PPO) model passing the instruct-fine-tuned PEFT model to it. PPO will be used to optimize the RL policy against the reward model.

In [18]:
ppo_model = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(peft_model,
                                                               torch_dtype=torch.bfloat16,
                                                               is_trainable=True)

print(f'PPO model parameters to be updated (ValueHead + 769 params):\n{print_number_of_trainable_model_parameters(ppo_model)}\n')
print(ppo_model.v_head)

PPO model parameters to be updated (ValueHead + 769 params):

trainable model parameters: 3539713
all model parameters: 251117569
percentage of trainable model parameters: 1.41%

ValueHead(
  (dropout): Dropout(p=0.1, inplace=False)
  (summary): Linear(in_features=768, out_features=1, bias=True)
  (flatten): Flatten(start_dim=1, end_dim=-1)
)


The model is like a student who already learned how to answer questions. PPO is the coach that tries to make the student’s answers less toxic. But instead of changing the whole brain, PPO mostly trains a small “judge button” called the **ValueHead** plus your trainable **PEFT/LoRA adapter**.

The ValueHead is like a tiny score predictor attached to the model. It tries to guess, “Was this answer good or bad based on the reward?” In your lab, the reward comes from the hate-speech/toxicity classifier, and TRL’s PPO setup supports training language models from a reward signal.

The formula means:
- (n+1)×m
- If n = 768 and m = 1, then:
  (768 + 1) * 1 = 769

In [19]:
ref_model = create_reference_model(ppo_model)

print(f'Reference model parameters to be updated:\n{print_number_of_trainable_model_parameters(ref_model)}\n')

# everthing should be set to prepare the reward model.

Reference model parameters to be updated:

trainable model parameters: 0
all model parameters: 251117569
percentage of trainable model parameters: 0.00%



###**Prepare Reward Model**

Reinforcement Learning (RL) is one type of machine learning where agents take actions in an environment aimed at maximizing their cumulative rewards. The agent's behavior is defined by the policy. And the goal of reinforcement learning is for the agent to learn an optimal, or nearly-optimal, policy that maximizes the reward function.

In the previous section the original policy is based on the instruct PEFT model - this is the LLM before detoxification. Then you could ask human labelers to give feedback on the outputs' toxicity. However, it can be expensive to use them for the entire fine-tuning process. A practical way to avoid that is to use a reward model encouraging the agent to detoxify the question answers. The intuitive approach would be to do some form of sentiment analysis across two classes (not-hate and hate) and give a higher reward if there is higher a chance of getting class not-hate as an output.

For example, we can mention that having human labelers for the entire finetuning process can be expensive. A practical way to avoid that is to use a **reward model**.

use feedback generated by a model

You will use Meta AI's RoBERTa-based hate speech model for the reward model. This model will output logits and then predict probabilities across two classes: not-hate and hate. The logits of the output not-hate will be taken as a positive reward. Then, the model will be fine-tuned with PPO using those reward values.

Create the instance of the required model class for the RoBERTa model. You also need to load a tokenizer to test the model. Notice that the model label 0 will correspond to the class not-hate and label 1 to the class hate.

In [20]:

from sentence_transformers import SentenceTransformer, util
import torch

quality_model_name = "sentence-transformers/all-MiniLM-L6-v2"
quality_model = SentenceTransformer(quality_model_name)
quality_model = quality_model.to("cuda")

print("Quality reward model loaded:", quality_model_name)

def compute_quality_reward(generated, reference):
    """
    Cosine similarity between generated answer and human reference answer.
    Range: -1 to 1. Higher means closer to the human answer.
    """
    embeddings = quality_model.encode(
        [generated, reference],
        convert_to_tensor=True
    )
    score = util.cos_sim(embeddings[0], embeddings[1])
    return score.item()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Quality reward model loaded: sentence-transformers/all-MiniLM-L6-v2


Take some non-toxic text, tokenize it, and pass it to the model. Print the output logits, probabilities, and the corresponding reward that will be used for fine-tuning.

In [21]:
reference_answer = (
    "Overfitting occurs when a model learns the training data too well "
    "including its noise, causing poor generalization to new unseen data. "
    "Solutions include regularization, dropout, and cross-validation."
)

good_answer = (
    "Overfitting is when a model memorizes training data instead of learning "
    "the real pattern. It performs well on training data but poorly on new data. "
    "We fix it with regularization or simpler models."
)

bad_answer = (
    "Overfitting is a type of neural network activation function used in "
    "convolutional layers to normalize batch distributions."
)

good_reward = compute_quality_reward(good_answer, reference_answer)
bad_reward  = compute_quality_reward(bad_answer,  reference_answer)

print(f"Good answer reward (should be high): {good_reward:.4f}")
print(f"Bad answer reward  (should be low):  {bad_reward:.4f}")
print(f"Variance signal gap:                 {abs(good_reward - bad_reward):.4f}")

Good answer reward (should be high): 0.8912
Bad answer reward  (should be low):  0.6262
Variance signal gap:                 0.2650


Let's show a toxic comment. This will have a low reward because it is more toxic.

In [22]:
def quality_pipe(query_response_pairs, reference_answers):
    """
    Replaces sentiment_pipe entirely.
    Scores each generated answer against its human reference answer.
    Returns list of [{"label": "good", "score": float}] per sample.
    """
    rewards = []
    for generated, reference in zip(query_response_pairs, reference_answers):
        score = compute_quality_reward(generated, reference)
        rewards.append([{"label": "good", "score": score}])
    return rewards

# Verify output format matches what PPO expects
test_output = quality_pipe(
    [good_answer, bad_answer],
    [reference_answer, reference_answer]
)
print("quality_pipe output format:")
for i, r in enumerate(test_output):
    print(f"  sample {i}: {r}")

quality_pipe output format:
  sample 0: [{'label': 'good', 'score': 0.8912407755851746}]
  sample 1: [{'label': 'good', 'score': 0.6262301802635193}]


Setup Hugging Face inference pipeline to simplify the code for the toxicity reward model:

The reward model gave the non-toxic text a very high **nothate probability of 0.9996** and a very low **hate probability of 0.0004**, so PPO would treat it as a good answer.  
For the toxic text, the model gave a high **hate probability of 0.9220** and a low **nothate probability of 0.0780**, so PPO would treat it as a bad answer.  
The key reward is the **nothate logit**: non-toxic = **4.1876**, toxic = **-1.3969**.

Clean answer = reward goes up.
Toxic answer = reward goes down.

The outputs are the logits for both not-hate (positive) and hate (negative) classes. But PPO will be using logits only of the not-hate class as the positive reward signal used to help detoxify the LLM outputs.

###**Evaluate Toxicity**

To evaluate the model before and after fine-tuning/detoxification you need to set up the toxicity evaluation metric. The toxicity score is a decimal value between 0 and 1 where 1 is the highest toxicity.

This cell creates an evaluator object that can measure toxicity across many generated answers, instead of manually running one sentence through the classifier.  This is useful because after detoxification, you need to compare the model’s outputs before and after training to see whether the toxicity score went down.

In [23]:
def evaluate_quality(model, tokenizer, dataset, num_samples):
    """
    Replaces evaluate_toxicity.
    Generates answers and scores them against human reference answers.
    Returns mean and std cosine similarity score.
    """
    max_new_tokens = 100
    quality_scores = []
    device = next(model.parameters()).device

    generation_config = GenerationConfig(
      max_new_tokens=max_new_tokens,
      top_k=0,
      top_p=1.0,
      do_sample=True
      )

    model.eval()

    for i, sample in tqdm(enumerate(dataset), total=num_samples):
        if i >= num_samples:
            break

        inputs = tokenizer(
            sample["query"],
            return_tensors="pt",
            padding=True
        ).to(device)

        with torch.no_grad():
            response_token_ids = model.generate(
                **inputs,
                generation_config=generation_config
            )

        generated_text = tokenizer.decode(
            response_token_ids[0],
            skip_special_tokens=True
        )

        # sample["answer"] is the human reference answer from your dataset
        reference_text = sample["answer"]
        score = compute_quality_reward(generated_text, reference_text)
        quality_scores.append(score)

    mean = np.mean(quality_scores)
    std  = np.std(quality_scores)
    return mean, std

This function loops through your dataset, gives each prompt to the model, generates an answer, and measures how toxic that generated answer is. The Python idea is building a reusable evaluation function instead of manually testing one answer at a time.

**Why this syntax is needed**:
The tokenizer prepares inputs for the model, and Hugging Face tokenizers return model inputs like input_ids and attention_mask. The .generate() method creates new text, and GenerationConfig controls generation settings like **max_new_tokens**, **sampling**, **top_k**, and **top_p**.

**How it fits your PPO lab:**
Before PPO, this function measures the model’s average toxicity. After PPO, you run it again to see whether the mean toxicity score went down.

For the DS interview assistant:
We can reuse the same structure later, but replace toxicity_evaluator with a quality evaluator that scores interview answers for correctness, clarity, completeness, and professional tone.

In [24]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

mean_before, std_before = evaluate_quality(
    model=ref_model,
    tokenizer=tokenizer,
    dataset=dataset["test"].with_format(type=None),
    num_samples=10
)

print(f"quality [mean, std] BEFORE PPO: [{mean_before:.4f}, {std_before:.4f}]")

100%|██████████| 10/10 [00:12<00:00,  1.24s/it]

quality [mean, std] BEFORE PPO: [0.6719, 0.1359]


**0.7514** mean **cosine similarity** is a strong starting point.
Your PEFT model from part 2 is already generating answers that are 75% semantically similar to the human reference answers. That is the payoff from  fine-tuning work in part 2. The model already knows the domain.
**0.0913** std is the number to watch.

###**Perform Fine-Tuning to Detoxify the Answers**
- Optimize a RL policy against the reward model using Proximal Policy Optimization (PPO).

##**Initialize PPOTrainer**
For the PPOTrainer initialization, you will need a collator. Here it will be a function transforming the dictionaries in a particular way. You can define and test it:

A collator is a helper function that takes a list of dataset rows and groups the values by column name. PPOTrainer needs this because it works with batches, and batches are easier to use when they look like {"query": [...], "input_ids": [...]} instead of a list of separate row dictionaries.

This cell defines and tests the batch-formatting function before giving it to PPOTrainer.


In [25]:
def collator(data):
    return dict((key, [d[key] for d in data]) for key in data[0])

test_data = [{"key1": "value1", "key2": "value2", "key3": "value3"}]

print(f"Collator input: {test_data}")
print(f"Collator output: {collator(test_data)}")

Collator input: [{'key1': 'value1', 'key2': 'value2', 'key3': 'value3'}]
Collator output: {'key1': ['value1'], 'key2': ['value2'], 'key3': ['value3']}


This creates the PPO training controller. The `ppo_model` is the model that will be updated, while `ref_model` stays frozen and acts like the **before detox** model.

**Why the reference model is needed:**
PPO compares the updated model against the frozen reference model using KL-divergence, so the model does not change too far away from its starting behavior. TRL’s PPO documentation says the reference model is used to compute a KL-divergence penalty that helps prevent the optimized model from drifting too far from the original language model.

**What the settings mean:**
- learning_rate = how big each training update is
- ppo_epochs = how many PPO passes per batch
- mini_batch_size = smaller chunks inside each PPO batch
- batch_size = how many prompts PPO uses at once


This is where you connect the trainable model, frozen reference model, tokenizer, training dataset, and collator together. After this, PPO can generate answers, score them with the toxicity reward model, and update ppo_model.

In [26]:
config = PPOConfig(
    model_name=model_name,
    learning_rate=1.41e-5,
    ppo_epochs=4,
    mini_batch_size=8,
    batch_size=32,
    init_kl_coef=0.2,
    target_kl=6.0,
    adap_kl_ctrl=True,
)

ppo_trainer = PPOTrainer(
    config=config,
    model=ppo_model,
    ref_model=ref_model,
    tokenizer=tokenizer,
    dataset=dataset["train"].with_format(type=None),
    data_collator=collator
)

###**Fine-Tune the Model**

The fine-tuning loop consists of the following main steps:
- Get the query responses from the policy LLM (PEFT model).
- Get sentiments for query/responses from hate speech RoBERTa model.
- Optimize policy with PPO using the (query, response, reward) triplet.

The operation is running if you see the following metrics appearing:
- objective/kl: minimize kl divergence,
- ppo/returns/mean: maximize mean returns,
- ppo/policy/advantages_mean: maximize advantages.

In [27]:
output_length_sampler = LengthSampler(50, 150)

generation_kwargs = {
    "min_length": 5,
    "top_k": 0,
    "top_p": 1.0,
    "do_sample": True
}

max_ppo_steps = 200        # was 10
device = next(ppo_model.parameters()).device

for step, batch in tqdm(enumerate(ppo_trainer.dataloader)):
    if step >= max_ppo_steps:
        break

    prompt_tensors = [
        torch.tensor(ids, dtype=torch.long).to(device)
        for ids in batch["input_ids"]
    ]

    response_tensors = []
    for prompt_tensor in prompt_tensors:
        max_new_tokens = output_length_sampler()
        generation_kwargs["max_new_tokens"] = max_new_tokens
        response = ppo_trainer.generate(prompt_tensor, **generation_kwargs)
        response_tensors.append(response.squeeze()[-max_new_tokens:])

    batch["answer"] = [
        tokenizer.decode(r.squeeze(), skip_special_tokens=True)
        for r in response_tensors
    ]

    query_response_pairs = [
        q + " " + r
        for q, r in zip(batch["query"], batch["answer"])
    ]

    # batch["answer"] is the human reference answer — already in your dataset
    rewards = quality_pipe(query_response_pairs, batch["answer"])

    reward_tensors = [
        torch.tensor(reward[0]["score"], device=device)
        for reward in rewards
    ]

    stats = ppo_trainer.step(prompt_tensors, response_tensors, reward_tensors)
    ppo_trainer.log_stats(stats, batch, reward_tensors)

    reward_mean = torch.stack(reward_tensors).mean().item()
    reward_std  = torch.stack(reward_tensors).std().item()

    print(
        f"Step {step+1:3d} | "
        f"KL: {stats['objective/kl']:.3f} | "
        f"reward mean: {reward_mean:.4f} | "
        f"reward std: {reward_std:.4f} | "
        f"advantages: {stats['ppo/policy/advantages_mean']:.2e}"
    )
    print("-" * 100)
    print("-" * 100)

1it [00:48, 48.79s/it]

Step   1 | KL: 14.868 | reward mean: 0.8388 | reward std: 0.0703 | advantages: -6.43e-10
----------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------


2it [01:30, 44.55s/it]

Step   2 | KL: 13.780 | reward mean: 0.8108 | reward std: 0.0843 | advantages: -7.15e-10
----------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------


3it [02:07, 41.28s/it]

...

Step  12 | KL: 14.350 | reward mean: 0.8080 | reward std: 0.0849 | advantages: -6.22e-09
----------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------


13it [08:50, 40.45s/it]

Step  13 | KL: 15.285 | reward mean: 0.8284 | reward std: 0.0690 | advantages: -5.17e-09
----------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------


14it [09:31, 40.79s/it]

Step  14 | KL: 12.050 | reward mean: 0.8348 | reward std: 0.0879 | advantages: 3.35e-09
----------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------


###**Evaluate the Model Quantitatively**
Load the PPO/PEFT model back in from disk and use the test dataset split to evaluate the toxicity score of the RL-fine-tuned model.

In [28]:
mean_after, std_after = evaluate_quality(
    model=ppo_model,
    tokenizer=tokenizer,
    dataset=dataset["test"].with_format(type=None),
    num_samples=10
)

print(f"quality [mean, std] AFTER PPO:  [{mean_after:.4f}, {std_after:.4f}]")
print(f"quality [mean, std] BEFORE PPO: [{mean_before:.4f}, {std_before:.4f}]")
print(f"mean improvement: {(mean_after - mean_before) / mean_before * 100:.2f}%")

100%|██████████| 10/10 [00:10<00:00,  1.06s/it]

quality [mean, std] AFTER PPO:  [0.7316, 0.1552]
quality [mean, std] BEFORE PPO: [0.6719, 0.1359]
mean improvement: 8.88%


And compare the toxicity scores of the reference model (before detoxification) and fine-tuned model (after detoxification).

###**Evaluate the Model Qualitatively**

Let's inspect some examples from the test dataset. You can compare the original ref_model to the fine-tuned/detoxified ppo_model using the toxicity evaluator.

In [29]:
batch_size = 20
compare_results = {}

df_batch = dataset["test"].with_format(type=None)[0:batch_size]
compare_results["query"]     = df_batch["query"]
compare_results["reference"] = df_batch["answer"]   # human answer for context
prompt_tensors = df_batch["input_ids"]

summary_tensors_ref = []
summary_tensors     = []

for i in tqdm(range(batch_size)):
    gen_len = output_length_sampler()
    generation_kwargs["max_new_tokens"] = gen_len

    summary = ref_model.generate(
        input_ids=torch.as_tensor(prompt_tensors[i]).unsqueeze(dim=0).to(device),
        **generation_kwargs
    ).squeeze()[-gen_len:]
    summary_tensors_ref.append(summary)

    summary = ppo_model.generate(
        input_ids=torch.as_tensor(prompt_tensors[i]).unsqueeze(dim=0).to(device),
        **generation_kwargs
    ).squeeze()[-gen_len:]
    summary_tensors.append(summary)

compare_results["answer_before"] = [
    tokenizer.decode(summary_tensors_ref[i]) for i in range(batch_size)
]
compare_results["answer_after"] = [
    tokenizer.decode(summary_tensors[i]) for i in range(batch_size)
]

# Score using quality reward against human reference
rewards_before = quality_pipe(
    [d + " " + s for d, s in zip(compare_results["query"], compare_results["answer_before"])],
    compare_results["reference"]
)
rewards_after = quality_pipe(
    [d + " " + s for d, s in zip(compare_results["query"], compare_results["answer_after"])],
    compare_results["reference"]
)

compare_results["reward_before"] = [r[0]["score"] for r in rewards_before]
compare_results["reward_after"]  = [r[0]["score"] for r in rewards_after]

pd.set_option("display.max_colwidth", 300)
df_compare = pd.DataFrame(compare_results)
df_compare["reward_diff"] = df_compare["reward_after"] - df_compare["reward_before"]
df_compare_sorted = df_compare.sort_values(by="reward_diff", ascending=False).reset_index(drop=True)
df_compare_sorted

100%|██████████| 20/20 [00:40<00:00,  2.04s/it]


,query,reference,answer_before,answer_after,reward_before,reward_after,reward_diff
0,Answer the following data science interview question clearly and concisely. What is logistic regression in classification? Answer: </s>,"Logistic regression in classification predicts the probability of a target variable belonging to a certain class, like estimating the likelihood of a loan being fraudulent.",<pad> Logical regression is a method used to evaluate the classification performance of different methods by dividing the classification data to their edges on the basis of known vertical or horizontal perpendicular differences.</s>,<pad> Logistic regression in classification involves maximizing the likelihood of encountering minor differences among classes by 91 percent.</s>,0.752305,0.800100,0.047795
1,Answer the following data science interview question clearly and concisely. What is the role of 'optimization algorithms' in machine learning? Answer: </s>,"Optimization algorithms in machine learning are used to minimize or maximize a function, which is often the loss function used to train a model.","<pad> The implementation of optimization algorithms in machine learning has a great potential for advancement in the field by leading to product-related improvements, such as possible European Union policies and reserves and EU policies via A knowledge pool.</s>","<pad> The purpose of optimization algorithms in machine learning is to reduce the risk of certain algorithms being used on real data, thus making them useful to maintaining learning efficiency.</s>",0.699279,0.742135,0.042856
2,Answer the following data science interview question clearly and concisely. What are the most used evaluation metrics for regression in machine learning? Answer: </s>,"The most commonly used evaluation metrics for regression are Mean absolute error (MAE), Mean squared error (MSE), Root mean squared error (RMSE), Root mean squared logarithmic error (RMSLE), Mean percentage error (MPE), Mean absolute percentage error (MAPE), and R2.","<pad> The most used evaluation metrics for regression in machine learning include the conditional fit, convergence, and tolerance.</s>","<pad> The evaluation metrics for regression are regression mean (those logarithmic differences between and across information expressed by the hypothesis, which often endorses the style of regression).</s>",0.673616,0.708598,0.034982
3,Answer the following data science interview question clearly and concisely. What is 'time series analysis' in machine learning? Answer: </s>,Time series analysis in machine learning involves analyzing time series data in order to extract meaningful statistics and characteristics of the data.,"<pad> Time series analysis in machine learning is an examination using stratological analysis, where the time series is calculated according to the race of the antecedent.</s>","<pad> Time series analysis in machine learning is the set of data that provides a model whose runtime introduces the knowledge to a fixed number of times, often utilizing applied linear reasoning to determine patterns in the data.</s>",0.798239,0.831689,0.033450
4,Answer the following data science interview question clearly and concisely. What is 'precision' in machine learning? Answer: </s>,"Precision in machine learning is a metric that calculates the accuracy of the positive predictions, i.e., the number of true positives divided by the total number of positive predictions (true positives + false positives).",<pad> Precision in machine learning usually refers to the ability to change the input data without increasing the accuracy of that data.</s>,"<pad> Precision in machine learning is the concept of determining the accuracy of a process, usually the result of analysis of key flags or errors.</s>",0.822734,0.842541,0.019807
5,Answer the following data science interview question clearly and concisely. ### Question: What is the curse of dimensionality? ### Answer: Answer: </s>,The curse of dimensio

Lab 3 demonstrated that PPO-based quality tuning requires a model
that is generating meaningfully suboptimal responses to provide
sufficient reward variance for policy learning. The Lab 2 PEFT model
had already converged to strong answer quality (0.75 cosine similarity),
leaving insufficient headroom for PPO to improve upon in 200 steps.

This is consistent with published RLHF literature showing diminishing
returns when the supervised fine-tuning baseline is already strong.
The Lab 2 PEFT checkpoint remains the deployment model of choice.



In [30]:
import os, datetime

paths = {
    "Lab 2 PEFT": "/content/drive/MyDrive/peft-ds-interview-checkpoint-local",
    "Lab 3 PPO":  "/content/drive/MyDrive/ppo-ds-interview-checkpoint",
}

for label, path in paths.items():
    exists = os.path.exists(path)
    if exists:
        mtime = os.path.getmtime(path + "/adapter_config.json")
        print(f"{label}: EXISTS — modified {datetime.datetime.fromtimestamp(mtime)}")
    else:
        print(f"{label}: MISSING")

Lab 2 PEFT: EXISTS — modified 2026-06-11 15:17:10
Lab 3 PPO: EXISTS — modified 2026-06-11 14:27:19


In [31]:
# Final cell — save PPO-tuned model to Drive

from google.colab import drive
drive.mount("/content/drive")

ppo_save_path = "/content/drive/MyDrive/ppo-ds-interview-checkpoint"

# NOTE: This model is saved for lab completion and comparison purposes only.
# Use /content/drive/MyDrive/peft-ds-interview-checkpoint-local for deployment.
# PPO did not improve answer quality — Lab 2 PEFT model is the better model.

# Save the PPO model (unwrap the ValueHead wrapper first)
ppo_model.pretrained_model.save_pretrained(ppo_save_path)
tokenizer.save_pretrained(ppo_save_path)

print("Saved PPO model to:", ppo_save_path)

# Verify immediately
import os
expected_files = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "tokenizer_config.json",
    "tokenizer.json",
]

print("\nVerifying checkpoint files:")
all_good = True
for f in expected_files:
    full_path = os.path.join(ppo_save_path, f)
    exists = os.path.exists(full_path)
    size   = os.path.getsize(full_path) if exists else 0
    status = "✓" if exists else "MISSING"
    print(f"  {f}: {status} ({size:,} bytes)")
    if not exists:
        all_good = False

if all_good:
    print("\nCheckpoint verified — safe to use in deployment.")
else:
    print("\nWARNING — missing files. Do not proceed to deployment.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Saved PPO model to: /content/drive/MyDrive/ppo-ds-interview-checkpoint

Verifying checkpoint files:
  adapter_config.json: ✓ (608 bytes)
  adapter_model.safetensors: ✓ (7,098,160 bytes)
  tokenizer_config.json: ✓ (20,771 bytes)
  tokenizer.json: ✓ (2,422,411 bytes)

Checkpoint verified — safe to use in deployment.


In [32]:
import os, datetime

paths = {
    "Lab 2 PEFT": "/content/drive/MyDrive/peft-ds-interview-checkpoint-local",
    "Lab 3 PPO":  "/content/drive/MyDrive/ppo-ds-interview-checkpoint",
}

for label, path in paths.items():
    exists = os.path.exists(path)
    if exists:
        mtime = os.path.getmtime(path + "/adapter_config.json")
        print(f"{label}: EXISTS — modified {datetime.datetime.fromtimestamp(mtime)}")
    else:
        print(f"{label}: MISSING")

Lab 2 PEFT: EXISTS — modified 2026-06-11 15:17:10
Lab 3 PPO: EXISTS — modified 2026-06-11 15:28:16


Ready for the deployment stage.
When you start that notebook, load from:

- pythonpeft_model_path = "/content/drive/MyDrive peft-ds-interview-checkpoint-local"